# Autoformer 추가 실험 (논문 1~2단계 재현)

메인 노트북(`Autoformer_from_scratch.ipynb`)에서 재현한 주 결과(Table 1·10) 외에, **직접 구현한 모델만으로 할 수 있는 논문의 나머지 실험**을 재현합니다.

| 순서 | 실험 | 논문 | 내용 | 기본 반복 | 예상 시간 (A100) |
|---|---|---|---|---|---|
| 1 | `fig7` | Figure 7 | 출력 길이별 메모리 · 실행 시간 (Auto-Correlation vs Attention 계열) | – | 약 5~10분 |
| 2 | `table4` | Table 4 | Auto-Correlation을 Full / LogSparse / LSH / ProbSparse Attention으로 교체 (ETTm2, 입력 96·192·336 × 예측 336·720·1440) | 1회 | 약 6~9시간 |
| 3 | `fig5` | Figure 5 | 학습된 의존성 시각화 (Table 4의 입력 96 → 예측 336 모델 사용) | – | 1분 |
| 4 | `table6` | Table 6 | 하이퍼파라미터 c = 1~5 민감도 (6개 데이터셋) | 3회 | 약 4시간 |
| 5 | `table7` | Table 7 | 입력 길이에 따른 성능 (ETTm2 · Electricity · ILI) | 3회 | 약 2.5시간 |
| 6 | `table8` | Table 8 | 디코더 입력에 붙이는 과거 길이 (0 / I/2 / I) | 3회 | 약 40분 |
| 7 | `table5` | Table 5 | ETT 4종(ETTh1·h2·m1·m2) 전체 벤치마크 | 3회 | 약 2시간 |
| 8 | `table2` | Table 2 | 단변량 예측 (ETTm2, Exchange) | 3회 | 약 40분 |

시간은 메인 실험(72회, 약 3.5시간)의 실측을 바탕으로 한 **대략적인 추정**입니다.

## 중간에 끊겨도 이어서 돌아갑니다

- **실험 1회가 끝날 때마다** 결과가 `OUT_DIR/results/<실험>.csv`에 한 줄씩 저장됩니다. 다시 실행하면 끝난 실험은 건너뜁니다.
- **epoch이 끝날 때마다** 학습 상태(가중치, 옵티마이저, early stopping 기록, 난수 상태)를 `OUT_DIR/checkpoints/`에 저장합니다. 실험 도중에 끊기면 **마지막으로 끝난 epoch 다음부터** 이어서 학습합니다. 실험이 끝나면 이 파일은 지웁니다.
- 메모리가 부족(OOM)한 설정은 `status=oom`으로 기록해서 다시 시도하지 않습니다 (논문 Table 4의 "-"와 같은 의미).
- **끊겼을 때 할 일**: 런타임을 다시 연결하고 **런타임 → 모두 실행**만 누르면 됩니다.

## 메인 노트북과 다른 점

- 모델·데이터 로더·학습 함수는 메인 노트북 코드를 **그대로 복사**해서 씁니다 (섹션 2).
- 그 위에 세 가지만 덧붙였습니다 (섹션 3): ① 단변량(`features='S'`) 지원, ② 디코더 과거 길이 0 지원(Table 8), ③ Auto-Correlation 자리에 다른 Attention을 끼우는 기능(Table 4).
- 이어서 하기를 위해 실험마다 시드를 `2021 + 반복 번호`로 정합니다 (메인 노트북은 예측 길이마다 2021로 한 번 설정).

## 0. 환경 설정

- `WORK_DIR`는 메인 노트북과 같은 폴더로 맞춰 주세요. 데이터는 `WORK_DIR/Autoformer/dataset`을 그대로 씁니다.
- `RUN`에서 돌릴 실험을 고를 수 있습니다. 위 표의 순서대로 실행됩니다.
- `RUNS`로 실험별 반복 횟수를 바꿀 수 있습니다. 이미 끝난 반복은 건너뛰므로, 나중에 숫자를 늘려 다시 실행하면 **추가분만** 돌아갑니다.

In [ ]:
import os, sys, subprocess

# ▼ 본인 폴더 이름으로 수정 (메인 노트북과 같게)
WORK_DIR = '/content/drive/MyDrive/황대경_Autoformer'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    WORK_DIR = os.environ.get('AUTOFORMER_WORK_DIR', WORK_DIR)

# True: 아주 작은 모델로 짧게 돌려서 모든 셀이 오류 없이 도는지만 확인 (결과 수치는 의미 없음, 별도 폴더에 저장)
QUICK_TEST = False
if os.environ.get('AUTOFORMER_QUICK') == '1':   # 로컬 테스트용
    QUICK_TEST = True

# 돌릴 실험 선택 (True만 실행)
RUN = {
    'fig7':   True,
    'table4': True,
    'fig5':   True,
    'table6': True,
    'table7': True,
    'table8': True,
    'table5': True,
    'table2': True,
}

# 실험별 반복 횟수 (나중에 늘리면 추가분만 실행)
RUNS = {'table4': 1, 'table6': 3, 'table7': 3, 'table8': 3, 'table5': 3, 'table2': 3}
if QUICK_TEST:
    RUNS = {k: 1 for k in RUNS}

NUM_WORKERS = int(os.environ.get('AUTOFORMER_WORKERS', '2'))

DATA_ROOT = os.path.join(WORK_DIR, 'Autoformer', 'dataset')
OUT_DIR = os.path.join(WORK_DIR, 'my_autoformer', 'extra_quick' if QUICK_TEST else 'extra')
RESULT_DIR = os.path.join(OUT_DIR, 'results')        # 실험 1회마다 한 줄씩 추가되는 결과 CSV
CKPT_DIR = os.path.join(OUT_DIR, 'checkpoints')      # epoch마다 저장되는 학습 상태 (이어서 하기용)
MODEL_DIR = os.path.join(OUT_DIR, 'models')          # 시각화용으로 남겨 두는 학습된 모델
FIG_DIR = os.path.join(OUT_DIR, 'figures')
for d in (RESULT_DIR, CKPT_DIR, MODEL_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

# LSH Attention(Table 4)에 필요. 공식 코드가 쓰는 라이브러리와 같은 것
try:
    import reformer_pytorch
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'reformer_pytorch==1.4.4'], check=True)
    import reformer_pytorch

print('DATA_ROOT 존재:', os.path.isdir(DATA_ROOT))
print('OUT_DIR:', OUT_DIR, '| QUICK_TEST:', QUICK_TEST)

## 1. 메인 노트북과 같은 코드

아래 셀들은 메인 노트북의 해당 셀을 **글자 하나 바꾸지 않고** 복사한 것입니다. 설명은 메인 노트북을 참고하세요.

| 셀 | 메인 노트북 섹션 |
|---|---|
| 라이브러리 · 시드 | 0 |
| 데이터 로더 | 1 |
| Series Decomposition | 2 |
| Auto-Correlation | 3 |
| 임베딩 | 4 |
| 인코더 · 디코더 | 5 |
| 전체 모델 · Config | 6 |
| 학습 보조 함수 | 8 (일부) |

In [ ]:
import math, time, copy, random, datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| torch', torch.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

def set_seed(seed=2021):
    # 공식 코드 run.py와 같은 시드(2021)를 사용
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def time_features(dates: pd.DatetimeIndex, freq: str) -> np.ndarray:
    # 공식 코드 utils/timefeatures.py와 같은 값을 직접 계산 → [T, 특징 수]
    feats = {
        'minute':     dates.minute / 59.0 - 0.5,
        'hour':       dates.hour / 23.0 - 0.5,
        'dayofweek':  dates.dayofweek / 6.0 - 0.5,
        'dayofmonth': (dates.day - 1) / 30.0 - 0.5,
        'dayofyear':  (dates.dayofyear - 1) / 365.0 - 0.5,
    }
    order = {
        't': ['minute', 'hour', 'dayofweek', 'dayofmonth', 'dayofyear'],
        'h': ['hour', 'dayofweek', 'dayofmonth', 'dayofyear'],
        'd': ['dayofweek', 'dayofmonth', 'dayofyear'],
    }[freq]
    return np.stack([np.asarray(feats[k], dtype=np.float64) for k in order], axis=1)

FREQ_TO_NFEAT = {'t': 5, 'h': 4, 'd': 3}


class TimeSeriesDataset(Dataset):
    def __init__(self, csv_path, split, seq_len, label_len, pred_len,
                 split_type='custom', freq='h', target='OT'):
        assert split in ('train', 'val', 'test')
        self.seq_len, self.label_len, self.pred_len = seq_len, label_len, pred_len

        df = pd.read_csv(csv_path)
        # custom 데이터는 [date, 나머지 변수..., target] 순서로 정렬 (ETT는 OT가 이미 마지막 열)
        cols = [c for c in df.columns if c not in ('date', target)]
        df = df[['date'] + cols + [target]]
        n = len(df)

        # ---- 1-1. 분할 경계 ----
        if split_type in ('ETT_hour', 'ETT_minute'):
            unit = 24 if split_type == 'ETT_hour' else 24 * 4      # 하루당 시점 수
            train_end = 12 * 30 * unit
            val_end = train_end + 4 * 30 * unit
            test_end = val_end + 4 * 30 * unit
        else:
            num_train, num_test = int(n * 0.7), int(n * 0.2)
            train_end, val_end, test_end = num_train, n - num_test, n
        starts = {'train': 0, 'val': train_end - seq_len, 'test': val_end - seq_len}
        ends = {'train': train_end, 'val': val_end, 'test': test_end}
        b1, b2 = starts[split], ends[split]

        # ---- 1-2. 정규화: 학습 구간 통계만 사용 ----
        values = df.iloc[:, 1:].values.astype(np.float64)
        train_values = values[:train_end]
        self.mean = train_values.mean(axis=0)
        self.std = train_values.std(axis=0)          # 모집단 표준편차 (sklearn StandardScaler와 동일)
        self.std[self.std == 0] = 1.0
        data = (values - self.mean) / self.std

        # ---- 1-3. 시간 특징 ----
        dates = pd.to_datetime(df['date'].values[b1:b2])
        self.data_x = data[b1:b2]
        self.data_stamp = time_features(pd.DatetimeIndex(dates), freq)

    def __len__(self):
        return len(self.data_x) - self.seq_len - self.pred_len + 1

    def __getitem__(self, i):
        # ---- 1-4. 슬라이딩 윈도우 ----
        s_begin, s_end = i, i + self.seq_len                    # 인코더 입력 구간
        r_begin = s_end - self.label_len                        # 디코더 구간 시작 (입력과 48개 겹침)
        r_end = r_begin + self.label_len + self.pred_len
        x = self.data_x[s_begin:s_end]
        y = self.data_x[r_begin:r_end]
        x_mark = self.data_stamp[s_begin:s_end]
        y_mark = self.data_stamp[r_begin:r_end]
        return (torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32),
                torch.tensor(x_mark, dtype=torch.float32), torch.tensor(y_mark, dtype=torch.float32))


def make_loaders(cfg, num_workers=2):
    # 공식 코드와 같은 설정: 학습·검증은 shuffle + drop_last, 테스트는 순서대로 전부 사용
    path = os.path.join(DATA_ROOT, cfg.data_path)
    sets = {s: TimeSeriesDataset(path, s, cfg.seq_len, cfg.label_len, cfg.pred_len,
                                 cfg.split_type, cfg.freq) for s in ('train', 'val', 'test')}
    loaders = {
        'train': DataLoader(sets['train'], cfg.batch_size, shuffle=True, drop_last=True, num_workers=num_workers),
        'val':   DataLoader(sets['val'],   cfg.batch_size, shuffle=True, drop_last=True, num_workers=num_workers),
        'test':  DataLoader(sets['test'],  cfg.batch_size, shuffle=False, drop_last=False, num_workers=num_workers),
    }
    return sets, loaders

In [ ]:
class MovingAvg(nn.Module):
    def __init__(self, kernel_size, stride=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):                                   # x: [B, L, D]
        pad = (self.kernel_size - 1) // 2
        front = x[:, :1, :].repeat(1, pad, 1)               # 첫 값 반복
        end = x[:, -1:, :].repeat(1, pad, 1)                # 마지막 값 반복
        x = torch.cat([front, x, end], dim=1)               # [B, L + 2*pad, D]
        return self.avg(x.permute(0, 2, 1)).permute(0, 2, 1)  # AvgPool1d는 [B, D, L] 입력


class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)          # X_t
        seasonal = x - trend                # X_s
        return seasonal, trend

In [ ]:
class AutoCorrelation(nn.Module):
    def __init__(self, factor=1):
        super().__init__()
        self.factor = factor          # 논문의 c
        self.record = False           # True면 지연별 상관을 저장 (섹션 11의 주기 분석용)
        self.last_corr = None

    def time_delay_agg_training(self, values, corr):
        # values, corr: [B, H, C, L]
        B, H, C, L = values.shape
        top_k = int(self.factor * math.log(L))
        mean_value = corr.mean(dim=1).mean(dim=1)                     # head·채널 평균 → [B, L]
        index = torch.topk(mean_value.mean(dim=0), top_k, dim=-1)[1]  # 배치 공통 지연 k개 → [k]
        weights = mean_value[:, index]                                # 샘플별 해당 지연의 상관 → [B, k]
        tmp_corr = torch.softmax(weights, dim=-1)
        out = torch.zeros_like(values)
        for i in range(top_k):
            pattern = torch.roll(values, shifts=-int(index[i]), dims=-1)   # 지연만큼 당겨서 정렬
            out = out + pattern * tmp_corr[:, i].view(B, 1, 1, 1)
        return out

    def time_delay_agg_inference(self, values, corr):
        B, H, C, L = values.shape
        top_k = int(self.factor * math.log(L))
        mean_value = corr.mean(dim=1).mean(dim=1)                     # [B, L]
        weights, delay = torch.topk(mean_value, top_k, dim=-1)        # 샘플별 지연 k개 → [B, k]
        tmp_corr = torch.softmax(weights, dim=-1)
        base = torch.arange(L, device=values.device).view(1, 1, 1, L)
        out = torch.zeros_like(values)
        for i in range(top_k):
            # 위치 t에 values[(t + τ) mod L]을 가져옴 = 샘플마다 다른 τ로 roll(-τ)한 것과 같음
            idx = (base + delay[:, i].view(B, 1, 1, 1)) % L
            pattern = torch.gather(values, -1, idx.expand(B, H, C, L))
            out = out + pattern * tmp_corr[:, i].view(B, 1, 1, 1)
        return out

    def forward(self, queries, keys, values):
        # queries: [B, L, H, E], keys/values: [B, S, H, E]
        B, L, H, E = queries.shape
        _, S, _, _ = values.shape
        if L > S:   # 3-5. 길이 맞추기
            zeros = torch.zeros_like(queries[:, :(L - S), :, :])
            values = torch.cat([values, zeros], dim=1)
            keys = torch.cat([keys, zeros], dim=1)
        else:
            values, keys = values[:, :L], keys[:, :L]

        # 3-2. FFT로 모든 지연의 자기상관 계산 (시간축을 마지막 차원으로)
        q_fft = torch.fft.rfft(queries.permute(0, 2, 3, 1).contiguous(), dim=-1)
        k_fft = torch.fft.rfft(keys.permute(0, 2, 3, 1).contiguous(), dim=-1)
        corr = torch.fft.irfft(q_fft * torch.conj(k_fft), n=L, dim=-1)    # [B, H, E, L]
        if self.record:
            self.last_corr = corr.mean(dim=1).mean(dim=1).detach().cpu()  # [B, L]

        # 3-3, 3-4. Time Delay Aggregation
        v = values.permute(0, 2, 3, 1).contiguous()
        agg = self.time_delay_agg_training(v, corr) if self.training else self.time_delay_agg_inference(v, corr)
        return agg.permute(0, 3, 1, 2).contiguous()                       # [B, L, H, E]


class AutoCorrelationLayer(nn.Module):
    # Multi-head 버전 (논문 식 7): Q, K, V 선형 사영 → head로 나눔 → Auto-Correlation → 합쳐서 출력 사영
    # self-attention의 AttentionLayer와 골격이 같으므로 그대로 갈아 끼울 수 있음
    def __init__(self, correlation, d_model, n_heads):
        super().__init__()
        d_head = d_model // n_heads
        self.inner_correlation = correlation
        self.query_projection = nn.Linear(d_model, d_head * n_heads)
        self.key_projection = nn.Linear(d_model, d_head * n_heads)
        self.value_projection = nn.Linear(d_model, d_head * n_heads)
        self.out_projection = nn.Linear(d_head * n_heads, d_model)
        self.n_heads = n_heads

    def forward(self, queries, keys, values):
        B, L, _ = queries.shape
        _, S, _ = keys.shape
        H = self.n_heads
        q = self.query_projection(queries).view(B, L, H, -1)
        k = self.key_projection(keys).view(B, S, H, -1)
        v = self.value_projection(values).view(B, S, H, -1)
        out = self.inner_correlation(q, k, v).view(B, L, -1)
        return self.out_projection(out)

In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super().__init__()
        self.tokenConv = nn.Conv1d(c_in, d_model, kernel_size=3, padding=1,
                                   padding_mode='circular', bias=False)
        nn.init.kaiming_normal_(self.tokenConv.weight, mode='fan_in', nonlinearity='leaky_relu')

    def forward(self, x):                                   # [B, L, c_in] → [B, L, d_model]
        return self.tokenConv(x.permute(0, 2, 1)).transpose(1, 2)


class TimeFeatureEmbedding(nn.Module):
    def __init__(self, d_model, freq='h'):
        super().__init__()
        self.embed = nn.Linear(FREQ_TO_NFEAT[freq], d_model, bias=False)

    def forward(self, x_mark):
        return self.embed(x_mark)


class DataEmbeddingWoPos(nn.Module):
    def __init__(self, c_in, d_model, freq='h', dropout=0.05):
        super().__init__()
        self.value_embedding = TokenEmbedding(c_in, d_model)
        self.temporal_embedding = TimeFeatureEmbedding(d_model, freq)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, x_mark):
        return self.dropout(self.value_embedding(x) + self.temporal_embedding(x_mark))

In [ ]:
class MyLayerNorm(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layernorm = nn.LayerNorm(channels)

    def forward(self, x):
        x_hat = self.layernorm(x)
        return x_hat - x_hat.mean(dim=1, keepdim=True)     # 시간축 평균 제거


class EncoderLayer(nn.Module):
    def __init__(self, attention, d_model, d_ff=2048, moving_avg=25, dropout=0.05, activation='gelu'):
        super().__init__()
        self.attention = attention
        self.conv1 = nn.Conv1d(d_model, d_ff, kernel_size=1, bias=False)
        self.conv2 = nn.Conv1d(d_ff, d_model, kernel_size=1, bias=False)
        self.decomp1 = SeriesDecomp(moving_avg)
        self.decomp2 = SeriesDecomp(moving_avg)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == 'relu' else F.gelu

    def forward(self, x):
        x = x + self.dropout(self.attention(x, x, x))
        x, _ = self.decomp1(x)                                          # S^{l,1}, 추세 버림
        y = self.dropout(self.activation(self.conv1(x.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))
        x, _ = self.decomp2(x + y)                                      # S^{l,2}, 추세 버림
        return x


class Encoder(nn.Module):
    def __init__(self, attn_layers, norm_layer=None):
        super().__init__()
        self.attn_layers = nn.ModuleList(attn_layers)
        self.norm = norm_layer

    def forward(self, x):
        for layer in self.attn_layers:
            x = layer(x)
        return self.norm(x) if self.norm is not None else x


class DecoderLayer(nn.Module):
    def __init__(self, self_attention, cross_attention, d_model, c_out, d_ff=2048,
                 moving_avg=25, dropout=0.05, activation='gelu'):
        super().__init__()
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.conv1 = nn.Conv1d(d_model, d_ff, kernel_size=1, bias=False)
        self.conv2 = nn.Conv1d(d_ff, d_model, kernel_size=1, bias=False)
        self.decomp1 = SeriesDecomp(moving_avg)
        self.decomp2 = SeriesDecomp(moving_avg)
        self.decomp3 = SeriesDecomp(moving_avg)
        self.dropout = nn.Dropout(dropout)
        # 추세 사영: d_model → c_out (식 4의 W를 하나로 구현)
        self.projection = nn.Conv1d(d_model, c_out, kernel_size=3, stride=1, padding=1,
                                    padding_mode='circular', bias=False)
        self.activation = F.relu if activation == 'relu' else F.gelu

    def forward(self, x, cross):
        x = x + self.dropout(self.self_attention(x, x, x))             # inner Auto-Correlation
        x, trend1 = self.decomp1(x)
        x = x + self.dropout(self.cross_attention(x, cross, cross))     # encoder-decoder Auto-Correlation
        x, trend2 = self.decomp2(x)
        y = self.dropout(self.activation(self.conv1(x.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))
        x, trend3 = self.decomp3(x + y)
        residual_trend = trend1 + trend2 + trend3
        residual_trend = self.projection(residual_trend.permute(0, 2, 1)).transpose(1, 2)
        return x, residual_trend


class Decoder(nn.Module):
    def __init__(self, layers, norm_layer=None, projection=None):
        super().__init__()
        self.layers = nn.ModuleList(layers)
        self.norm = norm_layer
        self.projection = projection

    def forward(self, x, cross, trend):
        for layer in self.layers:
            x, residual_trend = layer(x, cross)
            trend = trend + residual_trend                              # 추세 누적
        if self.norm is not None:
            x = self.norm(x)
        if self.projection is not None:
            x = self.projection(x)                                      # 계절 성분 d_model → c_out
        return x, trend

In [ ]:
class Autoformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.label_len, self.pred_len = cfg.label_len, cfg.pred_len
        self.decomp = SeriesDecomp(cfg.moving_avg)
        self.enc_embedding = DataEmbeddingWoPos(cfg.enc_in, cfg.d_model, cfg.freq, cfg.dropout)
        self.dec_embedding = DataEmbeddingWoPos(cfg.dec_in, cfg.d_model, cfg.freq, cfg.dropout)
        self.encoder = Encoder(
            [EncoderLayer(AutoCorrelationLayer(AutoCorrelation(cfg.factor), cfg.d_model, cfg.n_heads),
                          cfg.d_model, cfg.d_ff, cfg.moving_avg, cfg.dropout, cfg.activation)
             for _ in range(cfg.e_layers)],
            norm_layer=MyLayerNorm(cfg.d_model))
        self.decoder = Decoder(
            [DecoderLayer(AutoCorrelationLayer(AutoCorrelation(cfg.factor), cfg.d_model, cfg.n_heads),
                          AutoCorrelationLayer(AutoCorrelation(cfg.factor), cfg.d_model, cfg.n_heads),
                          cfg.d_model, cfg.c_out, cfg.d_ff, cfg.moving_avg, cfg.dropout, cfg.activation)
             for _ in range(cfg.d_layers)],
            norm_layer=MyLayerNorm(cfg.d_model),
            projection=nn.Linear(cfg.d_model, cfg.c_out, bias=True))

    def forward(self, x_enc, x_mark_enc, x_mark_dec, return_parts=False):
        # x_enc: [B, 96, D], x_mark_enc: [B, 96, n_time], x_mark_dec: [B, 48+96, n_time]
        B, _, D = x_enc.shape
        # ---- 식 2: 디코더 입력 초기화 ----
        mean = x_enc.mean(dim=1, keepdim=True).repeat(1, self.pred_len, 1)
        zeros = torch.zeros(B, self.pred_len, D, device=x_enc.device)
        seasonal_init, trend_init = self.decomp(x_enc)
        trend_init = torch.cat([trend_init[:, -self.label_len:, :], mean], dim=1)       # [B, 144, D]
        seasonal_init = torch.cat([seasonal_init[:, -self.label_len:, :], zeros], dim=1)  # [B, 144, D]
        # ---- 인코더 ----
        enc_out = self.encoder(self.enc_embedding(x_enc, x_mark_enc))                  # [B, 96, d_model]
        # ---- 디코더 ----
        dec_in = self.dec_embedding(seasonal_init, x_mark_dec)                         # [B, 144, d_model]
        seasonal_part, trend_part = self.decoder(dec_in, enc_out, trend=trend_init)    # 둘 다 [B, 144, D]
        out = trend_part + seasonal_part
        if return_parts:
            return out[:, -self.pred_len:], seasonal_part[:, -self.pred_len:], trend_part[:, -self.pred_len:]
        return out[:, -self.pred_len:]                                                  # [B, 96, D]


class Config:
    # 공식 run.py 기본값 + 데이터셋별 설정을 담는 간단한 설정 객체
    def __init__(self, **kw):
        defaults = dict(seq_len=96, label_len=48, pred_len=96, enc_in=7, dec_in=7, c_out=7,
                        d_model=512, n_heads=8, e_layers=2, d_layers=1, d_ff=2048, moving_avg=25,
                        factor=1, dropout=0.05, activation='gelu', freq='h',
                        batch_size=32, learning_rate=1e-4, train_epochs=10, patience=3,
                        data_path='ETT-small/ETTm2.csv', split_type='ETT_minute')
        defaults.update(kw)
        self.__dict__.update(defaults)

    def __repr__(self):
        return 'Config(' + ', '.join(f'{k}={v}' for k, v in self.__dict__.items()) + ')'

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience, self.counter = patience, 0
        self.best_loss, self.best_state, self.best_epoch = None, None, 0
        self.early_stop = False

    def step(self, val_loss, model, epoch):
        if self.best_loss is None or val_loss <= self.best_loss:     # 개선(같아도 개선으로 봄, 공식과 동일)
            self.best_loss, self.best_epoch, self.counter = val_loss, epoch, 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f'    검증 손실 개선 → 가중치 저장 (epoch {epoch})')
        else:
            self.counter += 1
            print(f'    EarlyStopping counter: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True


def adjust_learning_rate(optimizer, epoch, base_lr):
    lr = base_lr * (0.5 ** (epoch - 1))
    for g in optimizer.param_groups:
        g['lr'] = lr
    return lr


def to_device(batch):
    return [b.float().to(DEVICE) for b in batch]


@torch.no_grad()
def evaluate_loss(model, loader, pred_len, max_batches=None):
    # 공식 vali(): 배치별 MSE의 단순 평균
    model.eval(); losses = []
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        x, y, xm, ym = to_device(batch)
        out = model(x, xm, ym)
        losses.append(F.mse_loss(out.cpu(), y[:, -pred_len:].cpu()).item())
    model.train()
    return float(np.average(losses))


@torch.no_grad()
def predict(model, loader, pred_len, max_batches=None):
    model.eval(); preds, trues = [], []
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        x, y, xm, ym = to_device(batch)
        preds.append(model(x, xm, ym).cpu().numpy())
        trues.append(y[:, -pred_len:].cpu().numpy())
    return np.concatenate(preds), np.concatenate(trues)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import zipfile, glob, json

def save_fig(name):
    path = os.path.join(FIG_DIR, f'{name}.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print('저장:', path)

## 2. 확장 ① 단변량 · ② 디코더 과거 길이 0

**① 단변량 (`features='S'`, Table 2)**: 목표 변수(`OT`) 하나만 입력하고 예측합니다. 메인 로더는 열마다 따로 정규화하고 목표 변수를 마지막 열에 두므로, 데이터셋을 만든 뒤 **마지막 열만 남기면** 목표 변수만으로 정규화한 것과 똑같습니다.

**② 디코더 과거 길이 0 (`label_len=0`, Table 8)**: 메인 모델의 `trend_init[:, -self.label_len:]`는 `label_len=0`일 때 파이썬 규칙상 `[:, -0:]` = **전체**가 되어 버립니다. 그래서 "마지막 n칸"을 안전하게 자르는 `last_n`으로 바꾼 `forward`를 씁니다. `label_len > 0`이면 메인 모델과 결과가 같습니다.

In [ ]:
def make_loaders_x(cfg, num_workers=NUM_WORKERS):
    # 메인 make_loaders + 단변량 지원
    sets, _ = make_loaders(cfg, num_workers=0)
    if getattr(cfg, 'features', 'M') == 'S':
        for ds in sets.values():
            ds.data_x = ds.data_x[:, -1:]          # 목표 변수(OT, 마지막 열)만 남김
            ds.mean, ds.std = ds.mean[-1:], ds.std[-1:]
    loaders = {
        'train': DataLoader(sets['train'], cfg.batch_size, shuffle=True, drop_last=True, num_workers=num_workers),
        'val':   DataLoader(sets['val'],   cfg.batch_size, shuffle=True, drop_last=True, num_workers=num_workers),
        'test':  DataLoader(sets['test'],  cfg.batch_size, shuffle=False, drop_last=False, num_workers=num_workers),
    }
    return sets, loaders


def last_n(x, n):
    # x[:, -n:]와 같지만 n=0이면 빈 텐서를 돌려줌
    return x[:, x.shape[1] - n:]

## 3. 확장 ③ Attention 계열 (Table 4, Figure 5·7)

논문 Table 4는 Autoformer의 **Auto-Correlation 자리에** 다른 Attention을 끼워 비교합니다. 나머지 구조(분해 블록, 디코더 추세 누적 등)는 그대로입니다. 메인 노트북의 `AutoCorrelationLayer`가 Q·K·V 사영 → 내부 메커니즘 → 출력 사영 골격이므로, **내부 메커니즘만 바꿔 끼웁니다.**

| 이름 | 원 논문 | 방식 | 이 노트북의 구현 |
|---|---|---|---|
| `autocorr` | Autoformer | 주기(지연 τ) 단위로 시계열 통째 연결, FFT | 메인 노트북 그대로 |
| `full` | Transformer | 모든 점 쌍의 유사도 (L × S 행렬) | 직접 구현 |
| `logsparse` | LogTrans | 각 점이 2의 거듭제곱 칸 떨어진 점만 봄 | 직접 구현 (마스크 방식) |
| `lsh` | Reformer | 해시로 비슷한 점끼리 묶어 묶음 안에서만 Attention | `reformer_pytorch.LSHSelfAttention` (공식 코드와 같은 라이브러리·설정: bucket 4, hash 4) |
| `prob` | Informer | 중요한 쿼리 u개만 계산, 나머지는 V의 평균 | 공식 코드 `ProbAttention`을 옮겨 적음 (mask 없음) |

**구현상 주의할 점**
- Autoformer는 디코더에서도 인과 마스크를 쓰지 않으므로, 모든 Attention을 **마스크 없이**(양방향) 씁니다.
- `logsparse`는 원 논문(LogTrans)의 인과·합성곱 Attention을 그대로 옮긴 것이 아니라, **"거리 0, 1, 2, 4, 8, … 칸만 본다"는 희소 패턴만** 마스크로 구현했습니다. 계산은 전체 행렬로 하므로 **메모리 이점은 재현하지 않습니다** (Figure 7에서 Full과 메모리가 같게 나옵니다). 디코더→인코더(cross)에서는 디코더 위치를 실제 시각(`seq_len − label_len + i`)으로 맞춘 뒤 거리를 잽니다. 볼 수 있는 점이 하나도 없는 쿼리는 시각이 가장 가까운 점 하나를 보게 합니다.
- `lsh`의 cross Attention은 라이브러리의 `keys` 인자로 디코더 쿼리와 인코더 키를 함께 해싱합니다. 길이를 bucket×2의 배수로 맞추려고 0을 덧붙이고, 덧붙인 칸은 마스크로 제외합니다.
- `prob`의 `factor`는 Informer 기본값 5를 씁니다(`PROB_FACTOR`). 공식 코드의 `.squeeze()`는 배치가 1개일 때 모양이 깨져서 `.squeeze(-2)`로 바꿨습니다.

In [ ]:
from reformer_pytorch import LSHSelfAttention

PROB_FACTOR = 5          # Informer 기본값
LSH_BUCKET, LSH_HASHES = 4, 4   # 공식 코드 ReformerLayer 기본값


class FullAttention(nn.Module):
    # Transformer의 점 대 점 Attention: softmax(QKᵀ/√E) V
    def __init__(self, dropout=0.05):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.record = False       # True면 마지막 쿼리의 Attention 분포를 저장 (Figure 5)
        self.last_attn = None

    def mask(self, scores):       # LogSparse에서 덮어씀
        return scores

    def forward(self, queries, keys, values):
        # queries: [B, L, H, E], keys/values: [B, S, H, E]
        B, L, H, E = queries.shape
        scores = torch.einsum('blhe,bshe->bhls', queries, keys) / math.sqrt(E)   # [B, H, L, S]
        scores = self.mask(scores)
        A = torch.softmax(scores, dim=-1)
        if self.record:
            self.last_attn = A[:, :, -1, :].mean(dim=1).detach().cpu()        # [B, S]
        A = self.dropout(A)
        return torch.einsum('bhls,bshd->blhd', A, values).contiguous()          # [B, L, H, E]


class LogSparseAttention(FullAttention):
    # 위치 i는 거리 |i − j| ∈ {0, 1, 2, 4, 8, ...}인 j만 봄 (LogTrans의 희소 패턴)
    def __init__(self, dropout=0.05, query_offset=0):
        super().__init__(dropout)
        self.query_offset = query_offset   # cross: 디코더 i번째 = 시각 (seq_len − label_len + i)
        self._cache = {}

    def allowed(self, L, S, device):
        key = (L, S, str(device))
        if key not in self._cache:
            qt = torch.arange(L, device=device).view(L, 1) + self.query_offset
            kt = torch.arange(S, device=device).view(1, S)
            d = (qt - kt).abs()
            ok = d == 0
            p = 1
            while p <= L + S + self.query_offset:
                ok = ok | (d == p)
                p *= 2
            nearest = (qt.clamp(0, S - 1) == kt)                    # 볼 점이 없을 때 대비
            ok = ok | (nearest & ~ok.any(dim=1, keepdim=True))
            self._cache[key] = ok
        return self._cache[key]

    def mask(self, scores):
        L, S = scores.shape[-2:]
        return scores.masked_fill(~self.allowed(L, S, scores.device), float('-inf'))


class ProbAttention(nn.Module):
    # Informer의 ProbSparse Attention (공식 코드 layers/SelfAttention_Family.py, mask_flag=False)
    def __init__(self, factor=5, dropout=0.05):
        super().__init__()
        self.factor = factor
        self.record = False
        self.last_attn = None

    def _prob_QK(self, Q, K, sample_k, n_top):
        B, H, L_K, E = K.shape
        _, _, L_Q, _ = Q.shape
        K_expand = K.unsqueeze(-3).expand(B, H, L_Q, L_K, E)
        index_sample = torch.randint(L_K, (L_Q, sample_k), device=K.device)
        K_sample = K_expand[:, :, torch.arange(L_Q, device=K.device).unsqueeze(1), index_sample, :]
        Q_K_sample = torch.matmul(Q.unsqueeze(-2), K_sample.transpose(-2, -1)).squeeze(-2)
        M = Q_K_sample.max(-1)[0] - torch.div(Q_K_sample.sum(-1), L_K)       # 쿼리의 '뾰족함'
        M_top = M.topk(n_top, sorted=False)[1]
        Q_reduce = Q[torch.arange(B)[:, None, None], torch.arange(H)[None, :, None], M_top, :]
        return torch.matmul(Q_reduce, K.transpose(-2, -1)), M_top

    def forward(self, queries, keys, values):
        B, L_Q, H, D = queries.shape
        _, L_K, _, _ = keys.shape
        q, k, v = queries.transpose(2, 1), keys.transpose(2, 1), values.transpose(2, 1)   # [B, H, L, D]
        U_part = min(self.factor * int(np.ceil(np.log(L_K))), L_K)
        u = min(self.factor * int(np.ceil(np.log(L_Q))), L_Q)
        scores_top, index = self._prob_QK(q, k, sample_k=U_part, n_top=u)
        scores_top = scores_top / math.sqrt(D)
        context = v.mean(dim=-2, keepdim=True).expand(B, H, L_Q, D).clone()   # 게으른 쿼리 = V 평균
        attn = torch.softmax(scores_top, dim=-1)
        context[torch.arange(B)[:, None, None], torch.arange(H)[None, :, None], index, :] = \
            torch.matmul(attn, v).type_as(context)
        if self.record:   # 마지막 쿼리의 분포: 선택됐으면 그 행, 아니면 균등
            full = torch.full((B, H, L_K), 1.0 / L_K, device=v.device)
            hit = index == (L_Q - 1)                                           # [B, H, u]
            for b in range(B):
                for h in range(H):
                    pos = hit[b, h].nonzero()
                    if len(pos):
                        full[b, h] = attn[b, h, pos[0, 0]]
            self.last_attn = full.mean(dim=1).detach().cpu()
        return context.transpose(2, 1).contiguous()                             # [B, L, H, D]


class LSHLayer(nn.Module):
    # Reformer의 LSH Attention. 사영까지 포함한 층 전체를 대신함 (공식 ReformerLayer와 같은 방식)
    def __init__(self, d_model, n_heads, bucket_size=LSH_BUCKET, n_hashes=LSH_HASHES):
        super().__init__()
        self.bucket_size = bucket_size
        self.attn = LSHSelfAttention(dim=d_model, heads=n_heads, bucket_size=bucket_size, n_hashes=n_hashes)
        self.record = False
        self.last_attn = None

    def _pad(self, x):
        B, N, C = x.shape
        m = self.bucket_size * 2
        fill = (m - N % m) % m
        mask = torch.ones(B, N + fill, dtype=torch.bool, device=x.device)
        if fill:
            x = torch.cat([x, x.new_zeros(B, fill, C)], dim=1)
            mask[:, N:] = False
        return x, mask

    def forward(self, queries, keys, values):
        B, N, _ = queries.shape
        q, q_mask = self._pad(queries)
        if self.record:
            self.attn.lsh_attn._return_attn = True
            S = keys.shape[1]
            def cb(attn, buckets):
                a = attn.mean(dim=1)[:, N - 1, :]            # 마지막 쿼리 → [B, 전체 길이]
                self.last_attn = (a[:, :N] if keys is queries else a[:, q.shape[1]:q.shape[1] + S]).detach().cpu()
            self.attn.callback = cb
        if keys is queries:                                  # self-attention
            out = self.attn(q, input_mask=q_mask)
        else:                                                # cross-attention
            k, k_mask = self._pad(keys)
            out = self.attn(q, keys=k, input_mask=q_mask, context_mask=k_mask)
        return out[:, :N, :]


ATTN_TYPES = ['autocorr', 'full', 'logsparse', 'lsh', 'prob']
ATTN_LABEL = {'autocorr': 'Auto-Correlation', 'full': 'Full Attention', 'logsparse': 'LogSparse Attention',
              'lsh': 'LSH Attention', 'prob': 'ProbSparse Attention'}


def make_attention(cfg, kind):
    # kind: 'self' 또는 'cross'
    a = getattr(cfg, 'attn', 'autocorr')
    if a == 'lsh':
        return LSHLayer(cfg.d_model, cfg.n_heads)
    if a == 'autocorr':
        inner = AutoCorrelation(cfg.factor)
    elif a == 'full':
        inner = FullAttention(cfg.dropout)
    elif a == 'logsparse':
        inner = LogSparseAttention(cfg.dropout, query_offset=(cfg.seq_len - cfg.label_len) if kind == 'cross' else 0)
    elif a == 'prob':
        inner = ProbAttention(PROB_FACTOR, cfg.dropout)
    else:
        raise ValueError(a)
    return AutoCorrelationLayer(inner, cfg.d_model, cfg.n_heads)


class AutoformerX(Autoformer):
    # 메인 Autoformer + (③) Attention 교체 + (②) label_len=0 지원
    def __init__(self, cfg):
        super().__init__(cfg)
        if getattr(cfg, 'attn', 'autocorr') != 'autocorr':
            for layer in self.encoder.attn_layers:
                layer.attention = make_attention(cfg, 'self')
            for layer in self.decoder.layers:
                layer.self_attention = make_attention(cfg, 'self')
                layer.cross_attention = make_attention(cfg, 'cross')

    def forward(self, x_enc, x_mark_enc, x_mark_dec, return_parts=False):
        B, _, D = x_enc.shape
        mean = x_enc.mean(dim=1, keepdim=True).repeat(1, self.pred_len, 1)
        zeros = torch.zeros(B, self.pred_len, D, device=x_enc.device)
        seasonal_init, trend_init = self.decomp(x_enc)
        trend_init = torch.cat([last_n(trend_init, self.label_len), mean], dim=1)
        seasonal_init = torch.cat([last_n(seasonal_init, self.label_len), zeros], dim=1)
        enc_out = self.encoder(self.enc_embedding(x_enc, x_mark_enc))
        dec_in = self.dec_embedding(seasonal_init, x_mark_dec)
        seasonal_part, trend_part = self.decoder(dec_in, enc_out, trend=trend_init)
        out = trend_part + seasonal_part
        if return_parts:
            return out[:, -self.pred_len:], seasonal_part[:, -self.pred_len:], trend_part[:, -self.pred_len:]
        return out[:, -self.pred_len:]

**확인**: 다섯 가지 메커니즘 모두 같은 입력에서 출력 모양이 같아야 하고, `autocorr`·`label_len>0`일 때는 메인 `Autoformer`와 출력이 **완전히 같아야** 합니다(같은 가중치).

In [ ]:
_kw = dict(seq_len=96, label_len=48, pred_len=96, enc_in=7, dec_in=7, c_out=7, d_model=64, d_ff=128, freq='t', dropout=0.0)
_x = torch.randn(4, 96, 7); _xm = torch.rand(4, 96, 5) - 0.5; _ym = torch.rand(4, 144, 5) - 0.5
torch.manual_seed(0); _ref = Autoformer(Config(**_kw)).eval()
_mine = AutoformerX(Config(**_kw, attn='autocorr')).eval(); _mine.load_state_dict(_ref.state_dict())
with torch.no_grad():
    print('autocorr: 메인 Autoformer와 출력 일치 =', torch.allclose(_ref(_x, _xm, _ym), _mine(_x, _xm, _ym), atol=1e-6))
for _a in ATTN_TYPES:
    _m = AutoformerX(Config(**_kw, attn=_a))
    _out = _m(_x, _xm, _ym)
    _out.mean().backward()       # 역전파까지 되는지 확인
    print(f'{_a:10s} 출력 {tuple(_out.shape)} | 파라미터 {sum(p.numel() for p in _m.parameters()):,}')
_m0 = AutoformerX(Config(**{**_kw, 'label_len': 0}))
print('label_len=0 출력:', tuple(_m0(_x, _xm, _ym[:, 48:]).shape))
del _ref, _mine, _m, _m0

## 4. 이어서 할 수 있는 학습 · 기록

**결과 기록 (`ResultStore`)**: 실험마다 CSV 하나(`results/<실험>.csv`). 실험 1회가 끝나면 한 줄 추가합니다. 키(`key`)가 이미 있으면 건너뜁니다.

**학습 상태 저장 (`train_resumable`)**: 메인 노트북의 학습 루프와 같은 규칙(Adam 1e-4, MSE, epoch마다 학습률 절반, 검증 손실 기준 early stopping patience 3, 최고 검증 가중치로 테스트)에, epoch마다 다음을 `checkpoints/<키>.pt`로 저장하는 부분만 더했습니다.

| 저장 항목 | 이유 |
|---|---|
| 모델 가중치, Adam 상태 | 학습을 그대로 이어가기 위해 |
| early stopping 기록 (최고 검증 손실, 최고 가중치, 카운터) | 끊기기 전과 같은 기준으로 멈추고, 같은 최고 모델로 테스트하기 위해 |
| 다음 epoch 번호, 지금까지 학습 시간 | 이어서 셀 위치와 시간 기록 |
| 난수 상태 (torch · CUDA · numpy · random) | 배치 섞는 순서와 dropout을 끊기지 않은 것과 같게 이어가기 위해 |

저장은 임시 파일에 쓴 뒤 이름을 바꾸는 방식이라, 저장 도중에 끊겨도 이전 체크포인트가 깨지지 않습니다.

**테스트 지표**는 예측을 모두 모아 두지 않고 배치마다 오차 합을 누적해서 계산합니다(예측 1440처럼 길 때 메모리 절약). 값은 메인 노트북 방식(전체 평균)과 같습니다.

In [ ]:
import pandas as pd

class ResultStore:
    def __init__(self, exp):
        self.path = os.path.join(RESULT_DIR, f'{exp}.csv')

    def df(self):
        return pd.read_csv(self.path) if os.path.exists(self.path) else pd.DataFrame()

    def done_keys(self):
        d = self.df()
        return set(d['key']) if len(d) else set()

    def add(self, row):
        pd.DataFrame([row]).to_csv(self.path, mode='a', index=False, header=not os.path.exists(self.path))


def safe_name(key):
    return ''.join(ch if ch.isalnum() or ch in '-_.' else '_' for ch in key)


def _rng_state():
    st = {'torch': torch.get_rng_state(), 'numpy': np.random.get_state(), 'random': random.getstate()}
    if torch.cuda.is_available():
        st['cuda'] = torch.cuda.get_rng_state_all()
    return st


def _set_rng_state(st):
    torch.set_rng_state(st['torch']); np.random.set_state(st['numpy']); random.setstate(st['random'])
    if torch.cuda.is_available() and 'cuda' in st:
        torch.cuda.set_rng_state_all(st['cuda'])


def _atomic_save(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)


@torch.no_grad()
def test_metrics(model, loader, pred_len, max_batches=None):
    model.eval(); se = ae = 0.0; n = 0
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        x, y, xm, ym = to_device(batch)
        diff = (model(x, xm, ym) - y[:, -pred_len:]).double()
        se += (diff ** 2).sum().item(); ae += diff.abs().sum().item(); n += diff.numel()
    return se / n, ae / n


def train_resumable(cfg, loaders, key, seed, max_steps=None, max_eval_batches=None):
    ckpt_path = os.path.join(CKPT_DIR, safe_name(key) + '.pt')
    set_seed(seed)
    model = AutoformerX(cfg).float().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    criterion = nn.MSELoss()
    early = EarlyStopping(cfg.patience)
    start_epoch, elapsed, resumed = 1, 0.0, False

    if os.path.exists(ckpt_path):                       # ← 끊겼던 실험 이어서
        st = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(st['model']); optimizer.load_state_dict(st['optimizer'])
        early.__dict__.update(st['early'])
        start_epoch, elapsed, resumed = st['next_epoch'], st['elapsed'], True
        _set_rng_state(st['rng'])
        print(f'    ↻ 체크포인트에서 이어서: epoch {start_epoch}부터 (최고 검증 {early.best_loss:.5f} @ epoch {early.best_epoch})')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    epoch = start_epoch - 1
    for epoch in range(start_epoch, cfg.train_epochs + 1):
        t0 = time.time()
        lr = adjust_learning_rate(optimizer, epoch, cfg.learning_rate)   # epoch 1: 1e-4, 2: 5e-5, ...
        model.train(); losses = []
        for step, batch in enumerate(loaders['train']):
            if max_steps and step >= max_steps:
                break
            x, y, xm, ym = to_device(batch)
            optimizer.zero_grad()
            loss = criterion(model(x, xm, ym), y[:, -cfg.pred_len:])
            loss.backward(); optimizer.step()
            losses.append(loss.item())
        val_loss = evaluate_loss(model, loaders['val'], cfg.pred_len, max_eval_batches)
        elapsed += time.time() - t0
        print(f'    epoch {epoch} | lr {lr:.1e} | train {np.mean(losses):.5f} | val {val_loss:.5f} | {time.time() - t0:.0f}s')
        early.step(val_loss, model, epoch)
        if early.early_stop or epoch == cfg.train_epochs:
            break
        _atomic_save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'early': dict(early.__dict__),
                      'next_epoch': epoch + 1, 'elapsed': elapsed, 'rng': _rng_state()}, ckpt_path)

    model.load_state_dict(early.best_state)
    mse, mae = test_metrics(model, loaders['test'], cfg.pred_len, max_eval_batches)
    peak = torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else float('nan')
    info = dict(mse=mse, mae=mae, best_epoch=early.best_epoch, epochs_ran=epoch,
                train_time_sec=round(elapsed, 1), peak_mem_mb=round(peak, 0), resumed=resumed)
    return model, info, ckpt_path

### 데이터셋 설정과 실행 함수

데이터셋별 설정은 메인 노트북과 같이 공식 `scripts/`를 따릅니다. ETTh1·ETTh2·ETTm1(Table 5)과 단변량(Table 2) 설정도 공식 스크립트 값입니다.

| 데이터 | c (factor) | freq | epoch 상한 |
|---|---|---|---|
| ETTm2 | 1 (단변량은 3) | t | 10 (단변량 예측 192만 1) |
| ETTm1 | 3 | t | 10 |
| ETTh1, ETTh2 | 3 | h | 10 |
| Electricity | 3 | h | 10 |
| Exchange | 3 (단변량 예측 96·192는 1) | h | 예측 192·336은 1 (다변량) |
| Traffic | 3 | h | 3 |
| Weather | 3 | h | 예측 96만 2 |
| ILI | 3 | h | 10 |

In [ ]:
DATA = {
    'ETTh1':       dict(data_path='ETT-small/ETTh1.csv', split_type='ETT_hour', freq='h', enc_in=7),
    'ETTh2':       dict(data_path='ETT-small/ETTh2.csv', split_type='ETT_hour', freq='h', enc_in=7),
    'ETTm1':       dict(data_path='ETT-small/ETTm1.csv', split_type='ETT_minute', freq='t', enc_in=7),
    'ETTm2':       dict(data_path='ETT-small/ETTm2.csv', split_type='ETT_minute', freq='t', enc_in=7),
    'Electricity': dict(data_path='electricity/electricity.csv', split_type='custom', freq='h', enc_in=321),
    'Exchange':    dict(data_path='exchange_rate/exchange_rate.csv', split_type='custom', freq='h', enc_in=8),
    'Traffic':     dict(data_path='traffic/traffic.csv', split_type='custom', freq='h', enc_in=862),
    'Weather':     dict(data_path='weather/weather.csv', split_type='custom', freq='h', enc_in=21),
    'ILI':         dict(data_path='illness/national_illness.csv', split_type='custom', freq='h', enc_in=7),
}


def default_factor(name):
    return 1 if name == 'ETTm2' else 3


def default_epochs(name, pred_len):
    if name == 'Traffic':
        return 3
    if name == 'Exchange' and pred_len in (192, 336):
        return 1
    if name == 'Weather' and pred_len == 96:
        return 2
    return 10


def build_cfg(name, seq_len, label_len, pred_len, **over):
    d = dict(DATA[name])
    n_var = 1 if over.get('features', 'M') == 'S' else d['enc_in']
    d.update(seq_len=seq_len, label_len=label_len, pred_len=pred_len,
             enc_in=n_var, dec_in=n_var, c_out=n_var,
             factor=default_factor(name), train_epochs=default_epochs(name, pred_len))
    d.update(over)
    if QUICK_TEST:     # 셀 동작 확인용 초소형 설정
        d.update(d_model=32, d_ff=64, train_epochs=1, batch_size=8)
    return Config(**d)


def run_setting(exp, setting, cfg, runs, extra_cols, save_model_as=None):
    # setting: 설정을 나타내는 문자열, extra_cols: 결과 CSV에 함께 적을 열
    store = ResultStore(exp)
    todo = [r for r in range(runs) if f'{exp}|{setting}|run{r}' not in store.done_keys()]
    if not todo:
        return
    _, loaders = make_loaders_x(cfg)
    for r in todo:
        key = f'{exp}|{setting}|run{r}'
        print(f'===== {key} =====')
        row = dict(key=key, exp=exp, setting=setting, run=r, **extra_cols)
        try:
            model, info, ckpt = train_resumable(cfg, loaders, key, seed=2021 + r,
                                                max_steps=3 if QUICK_TEST else None,
                                                max_eval_batches=3 if QUICK_TEST else None)
            row.update(info, status='ok')
            print(f'  >>> MSE {info["mse"]:.4f} | MAE {info["mae"]:.4f} | best epoch {info["best_epoch"]} | '
                  f'{info["train_time_sec"]:.0f}s | peak {info["peak_mem_mb"]:.0f}MB')
            if save_model_as and r == 0:
                torch.save(model.state_dict(), os.path.join(MODEL_DIR, save_model_as))
            del model
        except torch.OutOfMemoryError:
            print('  >>> 메모리 부족(OOM): 기록하고 넘어감')
            row.update(status='oom'); ckpt = os.path.join(CKPT_DIR, safe_name(key) + '.pt')
        row['finished_at'] = datetime.datetime.now().isoformat(timespec='seconds')
        store.add(row)
        if os.path.exists(ckpt):
            os.remove(ckpt)                      # 결과를 기록했으니 학습 상태 파일은 삭제
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def summarize(exp, group_cols, paper=None, paper_key=None):
    # 반복 평균 ± 표준편차, 논문 값과 나란히
    d = ResultStore(exp).df()
    if not len(d):
        print('결과 없음'); return None
    ok = d[d.status == 'ok']
    g = ok.groupby(group_cols).agg(runs=('mse', 'count'), mse=('mse', 'mean'), mse_std=('mse', 'std'),
                                   mae=('mae', 'mean'), mae_std=('mae', 'std'),
                                   peak_mem_mb=('peak_mem_mb', 'mean')).reset_index()
    oom = d[d.status == 'oom'][group_cols].drop_duplicates()
    if len(oom):
        oom = oom.assign(runs=0, mse=np.nan, note='OOM')
        g = pd.concat([g, oom], ignore_index=True)
    if paper is not None:
        pm, pa = [], []
        for _, r in g.iterrows():
            v = paper.get(paper_key(r)) if paper_key else None
            pm.append(np.nan if v is None or v[0] is None else v[0]); pa.append(np.nan if v is None or v[1] is None else v[1])
        g['paper_mse'], g['paper_mae'] = pm, pa
        g['diff_mse'] = g['mse'] - g['paper_mse']
    g.to_csv(os.path.join(RESULT_DIR, f'{exp}_summary.csv'), index=False)
    with pd.option_context('display.float_format', '{:.3f}'.format, 'display.width', 200, 'display.max_rows', 200):
        display(g)
    return g


def progress(exp, total):
    n = len(ResultStore(exp).done_keys())
    print(f'[{exp}] 완료 {n} / {total}')

## 5. Figure 7 — 효율 분석

- **(a) 메모리**: Autoformer의 Auto-Correlation을 각 Attention으로 바꾸고, 입력 96에서 출력 길이를 192 → 3072로 두 배씩 늘리며 **학습 1 step(순전파+역전파)의 최대 GPU 메모리**를 잽니다. 논문은 배치 크기를 밝히지 않아 학습과 같은 32를 씁니다.
- **(b) 실행 시간**: 사영 등을 뺀 **메커니즘 자체**만 길이 L에서 반복 실행해 1회 평균 시간을 잽니다. 논문은 10³회 반복이라고 했는데, 긴 길이의 Full Attention은 오래 걸려 반복 횟수를 `TIME_REPS`로 조절할 수 있게 했습니다. 논문 그래프의 길이 축 값이 판독되지 않아 512 → 8192를 씁니다.
- 메모리가 모자라면 그 점은 비워 둡니다(OOM).
- GPU가 없으면 이 셀은 건너뜁니다.

In [ ]:
FIG7_OUT_LENS = [24, 48] if QUICK_TEST else [192, 384, 768, 1536, 3072]
FIG7_TIME_LENS = [64, 128] if QUICK_TEST else [512, 1024, 2048, 4096, 8192]
TIME_REPS = 3 if QUICK_TEST else 100


def fig7_memory():
    rows = []
    for a in ATTN_TYPES:
        for O in FIG7_OUT_LENS:
            cfg = build_cfg('ETTm2', 96, 48, O, attn=a)
            B = cfg.batch_size
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            try:
                model = AutoformerX(cfg).to(DEVICE)
                x = torch.randn(B, 96, 7, device=DEVICE); xm = torch.rand(B, 96, 5, device=DEVICE) - 0.5
                ym = torch.rand(B, 48 + O, 5, device=DEVICE) - 0.5; y = torch.randn(B, O, 7, device=DEVICE)
                F.mse_loss(model(x, xm, ym), y).backward()
                torch.cuda.synchronize()
                mem = torch.cuda.max_memory_allocated() / 2**20
            except torch.OutOfMemoryError:
                mem = np.nan
            rows.append(dict(attn=a, out_len=O, mem_mb=mem))
            print(f'{a:10s} O={O:5d} 메모리 {mem:,.0f} MB')
            model = x = y = None; torch.cuda.empty_cache()
    return pd.DataFrame(rows)


def fig7_time():
    rows = []
    B, H, E = 4, 8, 64
    for a in ATTN_TYPES:
        for L in FIG7_TIME_LENS:
            cfg = Config(d_model=H * E, n_heads=H, factor=1, dropout=0.0, seq_len=L, label_len=0, attn=a)
            layer = make_attention(cfg, 'self').to(DEVICE).eval()
            try:
                with torch.no_grad():
                    if a == 'lsh':
                        x = torch.randn(B, L, H * E, device=DEVICE); fn = lambda: layer(x, x, x)
                    else:
                        q = torch.randn(B, L, H, E, device=DEVICE); inner = layer.inner_correlation
                        fn = lambda: inner(q, q, q)
                    for _ in range(3): fn()                       # 예열
                    torch.cuda.synchronize(); t0 = time.time()
                    for _ in range(TIME_REPS): fn()
                    torch.cuda.synchronize()
                    ms = (time.time() - t0) / TIME_REPS * 1000
            except torch.OutOfMemoryError:
                ms = np.nan
            rows.append(dict(attn=a, length=L, ms=ms))
            print(f'{a:10s} L={L:5d} {ms:8.2f} ms')
            layer = None; torch.cuda.empty_cache()
    return pd.DataFrame(rows)


if RUN['fig7'] and torch.cuda.is_available():
    mem_path = os.path.join(RESULT_DIR, 'fig7_memory.csv'); time_path = os.path.join(RESULT_DIR, 'fig7_time.csv')
    mem_df = pd.read_csv(mem_path) if os.path.exists(mem_path) else fig7_memory(); mem_df.to_csv(mem_path, index=False)
    time_df = pd.read_csv(time_path) if os.path.exists(time_path) else fig7_time(); time_df.to_csv(time_path, index=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for a in ATTN_TYPES:
        m = mem_df[mem_df.attn == a]; t = time_df[time_df.attn == a]
        axes[0].plot(m.out_len, m.mem_mb / 1024, 'o-', label=ATTN_LABEL[a])
        axes[1].plot(t.length, t.ms, 'o-', label=ATTN_LABEL[a])
    axes[0].set(xscale='log', xlabel='output length (input 96)', ylabel='memory (GB)', title='(a) Memory, one training step')
    axes[1].set(xscale='log', yscale='log', xlabel='series length L', ylabel='time per run (ms)', title='(b) Running time of the mechanism')
    for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout(); save_fig('fig7_efficiency'); plt.show()
elif RUN['fig7']:
    print('GPU가 없어 Figure 7을 건너뜁니다.')

## 6. Table 4 — Auto-Correlation vs Attention 계열

- ETTm2 다변량, 입력 I ∈ {96, 192, 336} × 예측 O ∈ {336, 720, 1440}, 디코더 과거 길이 = I/2, c = 1 (메인 ETTm2 설정).
- 메커니즘 5종 × 설정 9개 = 45개. 기본 1회씩 (`RUNS['table4']`).
- 논문에서 Full Attention은 I=192·336의 O=1440에서 메모리 부족("-")이었습니다. 여기서도 메모리가 모자라면 OOM으로 기록합니다. A100 메모리가 논문 당시 GPU보다 커서 돌아갈 수도 있습니다.
- 입력 96 → 예측 336 모델은 Figure 5에 쓰려고 `models/`에 저장합니다.

In [ ]:
PAPER_T4 = {  # (attn, I, O): (MSE, MAE), None = 논문에서 OOM
    ('autocorr', 96, 336): (0.339, 0.372), ('autocorr', 96, 720): (0.422, 0.419), ('autocorr', 96, 1440): (0.555, 0.496),
    ('autocorr', 192, 336): (0.355, 0.392), ('autocorr', 192, 720): (0.429, 0.430), ('autocorr', 192, 1440): (0.503, 0.484),
    ('autocorr', 336, 336): (0.361, 0.406), ('autocorr', 336, 720): (0.425, 0.440), ('autocorr', 336, 1440): (0.574, 0.534),
    ('full', 96, 336): (0.375, 0.425), ('full', 96, 720): (0.537, 0.502), ('full', 96, 1440): (0.667, 0.589),
    ('full', 192, 336): (0.450, 0.470), ('full', 192, 720): (0.554, 0.533), ('full', 192, 1440): (None, None),
    ('full', 336, 336): (0.501, 0.485), ('full', 336, 720): (0.647, 0.491), ('full', 336, 1440): (None, None),
    ('logsparse', 96, 336): (0.362, 0.413), ('logsparse', 96, 720): (0.539, 0.522), ('logsparse', 96, 1440): (0.582, 0.529),
    ('logsparse', 192, 336): (0.420, 0.450), ('logsparse', 192, 720): (0.552, 0.513), ('logsparse', 192, 1440): (0.958, 0.736),
    ('logsparse', 336, 336): (0.474, 0.474), ('logsparse', 336, 720): (0.601, 0.524), ('logsparse', 336, 1440): (None, None),
    ('lsh', 96, 336): (0.366, 0.404), ('lsh', 96, 720): (0.502, 0.475), ('lsh', 96, 1440): (0.663, 0.567),
    ('lsh', 192, 336): (0.407, 0.421), ('lsh', 192, 720): (0.636, 0.571), ('lsh', 192, 1440): (1.069, 0.756),
    ('lsh', 336, 336): (0.442, 0.476), ('lsh', 336, 720): (0.615, 0.532), ('lsh', 336, 1440): (None, None),
    ('prob', 96, 336): (0.481, 0.472), ('prob', 96, 720): (0.822, 0.559), ('prob', 96, 1440): (0.715, 0.586),
    ('prob', 192, 336): (0.404, 0.425), ('prob', 192, 720): (1.148, 0.654), ('prob', 192, 1440): (0.732, 0.602),
    ('prob', 336, 336): (0.417, 0.434), ('prob', 336, 720): (0.631, 0.528), ('prob', 336, 1440): (1.133, 0.691),
}
T4_IN = [96, 192, 336]; T4_OUT = [336, 720, 1440]

if RUN['table4']:
    total = len(ATTN_TYPES) * len(T4_IN) * len(T4_OUT) * RUNS['table4']
    for I in T4_IN:
        for O in T4_OUT:
            for a in ATTN_TYPES:
                cfg = build_cfg('ETTm2', I, I // 2, O, attn=a)
                run_setting('table4', f'{a}_I{I}_O{O}', cfg, RUNS['table4'],
                            dict(dataset='ETTm2', attn=a, seq_len=I, pred_len=O),
                            save_model_as=f'table4_{a}_I{I}_O{O}.pth' if (I, O) == (96, 336) else None)
                progress('table4', total)
    t4 = summarize('table4', ['attn', 'seq_len', 'pred_len'], PAPER_T4,
                   lambda r: (r['attn'], int(r['seq_len']), int(r['pred_len'])))

In [ ]:
# 논문 Table 4 형식으로 보기 (MSE, 괄호 안은 논문 값)
if RUN['table4'] and t4 is not None:
    rows = []
    for a in ATTN_TYPES:
        row = {'mechanism': ATTN_LABEL[a]}
        for I in T4_IN:
            for O in T4_OUT:
                m = t4[(t4.attn == a) & (t4.seq_len == I) & (t4.pred_len == O)]
                p = PAPER_T4[(a, I, O)][0]
                mine = 'OOM' if len(m) and m.iloc[0].get('note') == 'OOM' else ('' if not len(m) else f'{m.iloc[0].mse:.3f}')
                row[f'I{I}-O{O}'] = f'{mine} ({p:.3f})' if p is not None else f'{mine} (-)'
        rows.append(row)
    display(pd.DataFrame(rows).set_index('mechanism'))

## 7. Figure 5 — 학습된 의존성 시각화

Table 4에서 저장한 **입력 96 → 예측 336** 모델로, 같은 테스트 샘플에 대해 **인코더 마지막 층**이 무엇을 보는지 비교합니다.

- **Auto-Correlation**: 이 샘플에서 자기상관이 큰 지연 τ 상위 6개(τ=0 제외)를 고르고, 마지막 시점에서 τ칸 앞 위치에 빨간 세로선을 긋습니다.
- **Attention 계열**: 마지막 시점(★)의 쿼리가 가장 크게 주목한 키 6개를 주황 점으로 표시합니다.

논문 그림처럼 Auto-Correlation이 고른 위치가 **주기(ETTm2는 96칸 = 하루) 위치**에 맞는지, Attention 계열은 흩어지는지를 봅니다. 그림은 마지막 변수(OT, 오일 온도)입니다.

In [ ]:
FIG5_SAMPLE = 0

@torch.no_grad()
def fig5():
    cfg0 = build_cfg('ETTm2', 96, 48, 336)
    sets, _ = make_loaders_x(cfg0, num_workers=0)
    x, y, xm, ym = [t.unsqueeze(0).to(DEVICE) for t in sets['test'][FIG5_SAMPLE]]
    series = x[0, :, -1].cpu().numpy(); L = len(series)
    avail = [a for a in ATTN_TYPES if os.path.exists(os.path.join(MODEL_DIR, f'table4_{a}_I96_O336.pth'))]
    if not avail:
        print('Table 4의 입력 96 → 예측 336 모델이 없습니다. Table 4를 먼저 실행하세요.'); return
    fig, axes = plt.subplots(1, len(avail), figsize=(4.2 * len(avail), 3), squeeze=False)
    for ax, a in zip(axes[0], avail):
        cfg = build_cfg('ETTm2', 96, 48, 336, attn=a)
        model = AutoformerX(cfg).to(DEVICE)
        model.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'table4_{a}_I96_O336.pth'), map_location=DEVICE))
        model.eval()
        att = model.encoder.attn_layers[-1].attention
        mech = att if a == 'lsh' else att.inner_correlation
        mech.record = True
        model(x, xm, ym)
        mech.record = False
        ax.plot(series, color='tab:blue', lw=1)
        ax.plot(L - 1, series[-1], marker='*', color='red', ms=12)
        if a == 'autocorr':
            corr = mech.last_corr[0].clone(); corr[0] = -float('inf')
            taus = torch.topk(corr, 6).indices.tolist()
            for tau in taus:
                ax.axvline(L - 1 - tau, color='red', lw=1, alpha=.7)
            ax.set_title(f'{ATTN_LABEL[a]}\nτ = {sorted(taus)}', fontsize=9)
        else:
            w = mech.last_attn[0][:L]
            top = torch.topk(w, 6).indices.numpy()
            ax.scatter(top, series[top], color='tab:orange', zorder=3, s=30)
            ax.set_title(f'{ATTN_LABEL[a]}\nkeys = {sorted(top.tolist())}', fontsize=9)
        ax.set_xlabel('input time step')
    plt.tight_layout(); save_fig('fig5_learned_dependencies'); plt.show()

if RUN['fig5']:
    fig5()

## 8. Table 6 — 하이퍼파라미터 c 민감도

c는 Auto-Correlation이 고르는 지연 개수 k = ⌊c · ln L⌋를 정합니다. 논문 설정대로 ILI는 입력 36 → 예측 48, 나머지는 입력 96 → 예측 336이고, c = 1~5를 바꿉니다. epoch 상한 등 나머지는 데이터셋 기본 설정을 따릅니다(예: Exchange 336은 1 epoch, Traffic은 3 epoch).

In [ ]:
PAPER_T6 = {  # (dataset, c): (MSE, MAE)
    ('ETTm2', 1): (0.339, 0.372), ('ETTm2', 2): (0.363, 0.389), ('ETTm2', 3): (0.339, 0.372), ('ETTm2', 4): (0.336, 0.369), ('ETTm2', 5): (0.410, 0.415),
    ('Electricity', 1): (0.252, 0.356), ('Electricity', 2): (0.224, 0.332), ('Electricity', 3): (0.231, 0.338), ('Electricity', 4): (0.232, 0.341), ('Electricity', 5): (0.273, 0.371),
    ('Exchange', 1): (0.511, 0.528), ('Exchange', 2): (0.511, 0.528), ('Exchange', 3): (0.509, 0.524), ('Exchange', 4): (0.513, 0.527), ('Exchange', 5): (0.517, 0.527),
    ('Traffic', 1): (0.706, 0.488), ('Traffic', 2): (0.673, 0.418), ('Traffic', 3): (0.619, 0.385), ('Traffic', 4): (0.607, 0.378), ('Traffic', 5): (0.618, 0.379),
    ('Weather', 1): (0.348, 0.388), ('Weather', 2): (0.358, 0.390), ('Weather', 3): (0.359, 0.395), ('Weather', 4): (0.349, 0.388), ('Weather', 5): (0.366, 0.399),
    ('ILI', 1): (2.754, 1.088), ('ILI', 2): (2.641, 1.072), ('ILI', 3): (2.669, 1.085), ('ILI', 4): (3.041, 1.178), ('ILI', 5): (3.076, 1.172),
}
T6_DATA = ['ETTm2', 'Exchange', 'ILI', 'Weather', 'Electricity', 'Traffic']   # 빠른 것부터
T6_C = [1, 2, 3, 4, 5]

if RUN['table6']:
    total = len(T6_DATA) * len(T6_C) * RUNS['table6']
    for name in T6_DATA:
        I, Lb, O = (36, 18, 48) if name == 'ILI' else (96, 48, 336)
        for c in T6_C:
            cfg = build_cfg(name, I, Lb, O, factor=c)
            run_setting('table6', f'{name}_c{c}', cfg, RUNS['table6'], dict(dataset=name, factor=c, seq_len=I, pred_len=O))
            progress('table6', total)
    t6 = summarize('table6', ['dataset', 'factor'], PAPER_T6, lambda r: (r['dataset'], int(r['factor'])))
    if t6 is not None:
        fig, axes = plt.subplots(1, len(T6_DATA), figsize=(3.2 * len(T6_DATA), 2.8))
        for ax, name in zip(axes, T6_DATA):
            s = t6[t6.dataset == name].sort_values('factor')
            ax.errorbar(s.factor, s.mse, yerr=s.mse_std, fmt='o-', label='mine', capsize=3)
            ax.plot(T6_C, [PAPER_T6[(name, c)][0] for c in T6_C], 's--', label='paper')
            ax.set(title=name, xlabel='c', xticks=T6_C); ax.grid(alpha=.3)
        axes[0].set_ylabel('MSE'); axes[0].legend(fontsize=8)
        plt.tight_layout(); save_fig('table6_factor_c'); plt.show()

## 9. Table 7 — 입력 길이

예측 길이를 고정(ILI 48, 나머지 336)하고 입력 길이를 바꿉니다. 디코더 과거 길이는 입력의 절반(I/2)입니다. 논문 표에 있는 ETTm2 · Electricity · ILI를 돌립니다.

In [ ]:
PAPER_T7 = {
    ('ETTm2', 96): (0.339, 0.372), ('ETTm2', 192): (0.355, 0.392), ('ETTm2', 336): (0.361, 0.406), ('ETTm2', 720): (0.419, 0.430),
    ('Electricity', 96): (0.231, 0.338), ('Electricity', 192): (0.200, 0.316), ('Electricity', 336): (0.225, 0.335), ('Electricity', 720): (0.226, 0.346),
    ('ILI', 24): (3.406, 1.247), ('ILI', 36): (2.669, 1.085), ('ILI', 48): (2.656, 1.075), ('ILI', 60): (2.779, 1.091),
}
T7_PLAN = [('ILI', [24, 36, 48, 60], 48), ('ETTm2', [96, 192, 336, 720], 336), ('Electricity', [96, 192, 336, 720], 336)]

if RUN['table7']:
    total = sum(len(Is) for _, Is, _ in T7_PLAN) * RUNS['table7']
    for name, Is, O in T7_PLAN:
        for I in Is:
            cfg = build_cfg(name, I, I // 2, O)
            run_setting('table7', f'{name}_I{I}', cfg, RUNS['table7'], dict(dataset=name, seq_len=I, pred_len=O))
            progress('table7', total)
    t7 = summarize('table7', ['dataset', 'seq_len'], PAPER_T7, lambda r: (r['dataset'], int(r['seq_len'])))
    if t7 is not None:
        fig, axes = plt.subplots(1, 3, figsize=(11, 2.8))
        for ax, (name, Is, O) in zip(axes, T7_PLAN):
            s = t7[t7.dataset == name].sort_values('seq_len')
            ax.errorbar(s.seq_len, s.mse, yerr=s.mse_std, fmt='o-', label='mine', capsize=3)
            ax.plot(Is, [PAPER_T7[(name, I)][0] for I in Is], 's--', label='paper')
            ax.set(title=f'{name} (predict {O})', xlabel='input length', xticks=Is); ax.grid(alpha=.3)
        axes[0].set_ylabel('MSE'); axes[0].legend(fontsize=8)
        plt.tight_layout(); save_fig('table7_input_length'); plt.show()

## 10. Table 8 — 디코더 입력에 붙이는 과거 길이

ETTm2 입력 96 → 예측 336에서 디코더 입력을 **예측 칸만(O)**, **과거 절반 + 예측(I/2 + O, 기본값)**, **과거 전체 + 예측(I + O)**으로 바꿉니다. 논문은 메모리도 함께 보고했으므로 학습 중 최대 GPU 메모리(`peak_mem_mb`)도 함께 봅니다. 논문의 메모리 값은 측정 방식(배치·GPU)이 달라 절대값보다 **증가 경향**을 비교하세요.

In [ ]:
PAPER_T8 = {0: (0.360, 0.383), 48: (0.339, 0.372), 96: (0.333, 0.369)}
PAPER_T8_MEM = {0: 3029, 48: 3271, 96: 3599}

if RUN['table8']:
    total = 3 * RUNS['table8']
    for Lb in [0, 48, 96]:
        cfg = build_cfg('ETTm2', 96, Lb, 336)
        run_setting('table8', f'ETTm2_label{Lb}', cfg, RUNS['table8'], dict(dataset='ETTm2', label_len=Lb, seq_len=96, pred_len=336))
        progress('table8', total)
    t8 = summarize('table8', ['label_len'], PAPER_T8, lambda r: int(r['label_len']))
    if t8 is not None:
        t8['paper_mem_mb'] = t8.label_len.map(PAPER_T8_MEM)
        display(t8[['label_len', 'mse', 'paper_mse', 'mae', 'paper_mae', 'peak_mem_mb', 'paper_mem_mb']])

## 11. Table 5 — ETT 4종 전체 벤치마크

ETTh1·ETTh2(1시간 간격)와 ETTm1·ETTm2(15분 간격)를 입력 96으로, 공식 스크립트의 예측 길이로 돌립니다. ETTm2의 예측 96은 메인 실험과 같은 설정이지만 비교를 위해 다시 돌립니다.

In [ ]:
PAPER_T5 = {
    ('ETTh1', 24): (0.384, 0.425), ('ETTh1', 48): (0.392, 0.419), ('ETTh1', 168): (0.490, 0.481), ('ETTh1', 336): (0.505, 0.484), ('ETTh1', 720): (0.498, 0.500),
    ('ETTh2', 24): (0.261, 0.341), ('ETTh2', 48): (0.312, 0.373), ('ETTh2', 168): (0.457, 0.455), ('ETTh2', 336): (0.471, 0.475), ('ETTh2', 720): (0.474, 0.484),
    ('ETTm1', 24): (0.383, 0.403), ('ETTm1', 48): (0.454, 0.453), ('ETTm1', 96): (0.481, 0.463), ('ETTm1', 288): (0.634, 0.528), ('ETTm1', 672): (0.606, 0.542),
    ('ETTm2', 24): (0.153, 0.261), ('ETTm2', 48): (0.178, 0.280), ('ETTm2', 96): (0.255, 0.339), ('ETTm2', 288): (0.342, 0.378), ('ETTm2', 672): (0.434, 0.430),
}
T5_PLAN = [('ETTh1', [24, 48, 168, 336, 720]), ('ETTh2', [24, 48, 168, 336, 720]),
           ('ETTm1', [24, 48, 96, 288, 672]), ('ETTm2', [24, 48, 96, 288, 672])]

if RUN['table5']:
    total = sum(len(Os) for _, Os in T5_PLAN) * RUNS['table5']
    for name, Os in T5_PLAN:
        for O in Os:
            cfg = build_cfg(name, 96, 48, O)
            run_setting('table5', f'{name}_O{O}', cfg, RUNS['table5'], dict(dataset=name, seq_len=96, pred_len=O))
            progress('table5', total)
    t5 = summarize('table5', ['dataset', 'pred_len'], PAPER_T5, lambda r: (r['dataset'], int(r['pred_len'])))

## 12. Table 2 — 단변량 예측

목표 변수(OT) 하나만으로 예측합니다. 설정은 공식 단변량 스크립트를 따릅니다.

| 데이터 | 디코더 과거 길이 | c | 특이 사항 |
|---|---|---|---|
| ETTm2 (`features=S`) | 96 | 3 | 예측 192만 1 epoch |
| Exchange (`features=S`) | 48 | 예측 96·192는 1, 336·720은 3 | epoch 상한 10 |

논문 본문 Table 2의 ETT 예측 336 값은 0.154입니다(본문 4.1절에는 0.145로 적혀 있어 불일치).

In [ ]:
PAPER_T2 = {
    ('ETTm2', 96): (0.065, 0.189), ('ETTm2', 192): (0.118, 0.256), ('ETTm2', 336): (0.154, 0.305), ('ETTm2', 720): (0.182, 0.335),
    ('Exchange', 96): (0.241, 0.387), ('Exchange', 192): (0.273, 0.403), ('Exchange', 336): (0.508, 0.539), ('Exchange', 720): (0.991, 0.768),
}

if RUN['table2']:
    total = 8 * RUNS['table2']
    for O in [96, 192, 336, 720]:
        cfg = build_cfg('Exchange', 96, 48, O, features='S', factor=1 if O in (96, 192) else 3, train_epochs=10)
        run_setting('table2', f'Exchange_S_O{O}', cfg, RUNS['table2'], dict(dataset='Exchange', features='S', seq_len=96, pred_len=O))
        progress('table2', total)
    for O in [96, 192, 336, 720]:
        cfg = build_cfg('ETTm2', 96, 96, O, features='S', factor=3, train_epochs=1 if O == 192 else 10)
        run_setting('table2', f'ETTm2_S_O{O}', cfg, RUNS['table2'], dict(dataset='ETTm2', features='S', seq_len=96, pred_len=O))
        progress('table2', total)
    t2 = summarize('table2', ['dataset', 'pred_len'], PAPER_T2, lambda r: (r['dataset'], int(r['pred_len'])))

## 13. 결과 묶기

`results/`의 CSV와 `figures/`의 그림을 `extra_results.zip` 하나로 묶습니다. 이 파일을 보내 주시면 README와 발표자료에 정리합니다. 실험이 일부만 끝났어도 그때까지의 결과가 들어갑니다.

In [ ]:
zip_path = os.path.join(OUT_DIR, 'extra_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(os.path.join(RESULT_DIR, '*.csv')) + glob.glob(os.path.join(FIG_DIR, '*.png')):
        z.write(p, os.path.relpath(p, OUT_DIR))
print('저장:', zip_path)
for exp in ['table4', 'table6', 'table7', 'table8', 'table5', 'table2']:
    d = ResultStore(exp).df()
    print(f'{exp:7s} 완료 {len(d)}개' + (f' (OOM {int((d.status == "oom").sum())})' if len(d) else ''))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass